# Project: Language Model Implementations

### Author: Ayomide Olumide-Attah
### Date: November 13th, 2026

**About**: This project provides implementations of n-gram language models. The NGramModel class provides the functionality for implementing the n-gram models, and the LIPNGramModel class extends the NGramModel class, using linear interpolation as a smoothing strategy.

## The NGramModel class

In [34]:
from functools import reduce
from collections import defaultdict
import random
import numpy as np
import itertools

class NGramModel:
    def __init__(self, n: int, corpus: list[str]):
        self.n = n
        self.corpus = corpus

        self.V = []
        self.V_ids = {}
        self.N = 0
        self.frequencies = []

        self.probabilities = []

        self.build_vocabulary()
        self.build_probabilities()

    def build_vocabulary(self):
        V_repeated = list(reduce(lambda x, y: x + y,
                               [sentence.split() for sentence in self.corpus]))
        V = list(set(V_repeated))
        N = len(V)

        V_ids = {V[i]: i for i in range(N)}

        freqs = np.zeros(N)
        for word in V_repeated:
            i = V_ids[word]
            freqs[i] += 1

        self.V = V
        self.N = len(V)
        self.V_ids = V_ids
        self.frequencies = freqs

    def build_k_gram_probs(self, k: int) -> list[list[float]]:
        if k == 1: return self.frequencies / self.N

        idx = k - 2
        pregram_freqs = defaultdict(int)
        k_gram_probs = {}

        for sentence in self.corpus:
            words = sentence.split()
            for m in range(len(words) - k + 1):
                pregram_tup = tuple(self.V_ids[words[j]] for j in range(m, m + k - 1))
                word_id = self.V_ids[words[m + k - 1]]

                pregram_freqs[pregram_tup] += 1
                if pregram_tup not in k_gram_probs:
                    k_gram_probs[pregram_tup] = np.zeros(self.N)
                k_gram_probs[pregram_tup][word_id] += 1

        for pregram_tup, count in pregram_freqs.items():
            if count > 0:
                k_gram_probs[pregram_tup] /= count

        return k_gram_probs

    def build_probabilities(self):
        self.probabilities = [self.build_k_gram_probs(k) for k in range(1, self.n + 1)]

    def generate_probs(self, context: list[str]) -> list[float]:
        if self.n == 1: return self.probabilities[0]

        k = min(len(context), self.n - 1)
        pregram_tup = tuple(self.V_ids[word] for word in context[-k:])

        k_gram_probs = self.probabilities[k]
        probs = k_gram_probs.get(pregram_tup, np.zeros(self.N))
        return probs

    def generate_random(self, context: list[str]) -> str:
        probs = self.generate_probs(context)
        if sum(probs) == 0:
            return ""
        else:
            next_word = random.choices(self.V, probs)[0]
            return next_word


    def generate_top_k(self, context: list[str], k: int) -> str:
        probs = self.generate_probs(context)
        if sum(probs) == 0:
            return ""
        else:
            cands = [self.V[i] for i in np.argpartition(probs, -k)[-k:] if probs[i] > 0]
            next_word = random.choice(cands)
            return next_word

    def generate_highest(self, context: list[str]) -> str:
        return self.generate_top_k(context, k=1)

    def generate_top_p(self, context: list[str], p: float) -> str:
        probs = self.generate_probs(context)
        if sum(probs) == 0:
            return ""
        else:
            ids = np.argsort(probs)
            words, props = [], []

            i = -1
            r = 0
            while r < p and i >= -len(ids):
                words.append(self.V[ids[i]])
                props.append(probs[ids[i]])

                r += probs[ids[i]]
                i -= 1

            next_word = random.choices(words, props)[0]
            return next_word


## The LIPNGramModel class

In [35]:
class LIPNGramModel(NGramModel):
    def __init__(self, n: int, corpus: list[str], params: list[float]):
        super().__init__(n, corpus)
        # NB: Assumes the length of 'params' is exactly 3
        self.params = params

    def generate_probs(self, context: list[str]) -> list[float]:
        k = min(len(context), self.n - 1)
        words = context[-k:]

        if k <= self.n - 2 or self.n < 3:
            return super().generate_probs(context)

        elif n == 3:
            pregram_id_1 = tuple(self.V_ids[word] for word in words)
            pregram_id_2 = tuple(self.V_ids[word] for word in words[1:])

            probs_1 = self.probabilities[2].get(pregram_id_1, np.zeros(self.N))
            probs_2 = self.probabilities[1].get(pregram_id_2, np.zeros(self.N))
            probs_3 = self.probabilities[0]

            probs = np.array(self.params) @ np.array([probs_1, probs_2, probs_3])
            return probs

        else:
            pregram_id_1 = tuple(self.V_ids[word] for word in words)
            pregram_id_2 = tuple(self.V_ids[word] for word in words[1:])
            pregram_id_3 = tuple(self.V_ids[word] for word in words[2:])

            probs_1 = self.probabilities[n - 1].get(pregram_id_1, np.zeros(self.N))
            probs_2 = self.probabilities[n - 2].get(pregram_id_2, np.zeros(self.N))
            probs_3 = self.probabilities[n - 3].get(pregram_id_3, np.zeros(self.N))

            probs = np.array(self.params) @ np.array([probs_1, probs_2, probs_3])
            return probs

## Corpus 1: A small example

In [36]:
corpus_1 = ["<s> I like strawberries </s>",
"<s> You like all strawberries </s>",
"<s> I hate sour fruits </s>",
"<s> You like all sweet fruits </s>",
"<s> I like chocolate covered strawberries </s>",
"<s> You like fruits </s>",
"<s> We like apples </s>"]

n = 2
model_1 = NGramModel(n, corpus_1)
lip_model_1 = LIPNGramModel(n, corpus_1, params=[0.7, 0.2, 0.1])

In [39]:
from IPython.display import clear_output
import time

context = ["<s>"]
for i in range(20):
    next_word = model_1.generate_random(context)
    if not next_word: break

    context.append(next_word)
    end = None if next_word != "</s>" else -1
    print(" ".join(context[1: end]))
    clear_output(wait=True)
    time.sleep(0.5)

We like chocolate covered strawberries


In [ ]:
context = ["<s>"]
for i in range(20):
    next_word = lip_model_1.generate_random(context)
    if not next_word: break

    context.append(next_word)
    end = None if next_word != "</s>" else -1
    print(" ".join(context[1: end]))
    clear_output(wait=True)
    time.sleep(0.5)

I like strawberries


## Corpus 2: A slightly bigger example

In [40]:
corpus_2 = [
    "<s> I love Math and he loves English </s>",
    "<s> He loves all subjects </s>",
    "<s> He loves Geography and she loves Math </s>",
    "<s> I love all science subjects </s>",
    "<s> We love science </s>",
    "<s> They love English </s>",
    "<s> I study Math and you study Geography </s>",
    "<s> You study all subjects </s>",
    "<s> She studies English </s>",
    "<s> They study Geography and she studies Math </s>",
    "<s> He studies Geography </s>",
    "<s> She studies all subjects </s>"
]

n = 3
model_2 = NGramModel(n, corpus_2)
lip_model_2 = LIPNGramModel(n, corpus_2, params=[0.9, 0.1, 0])

In [42]:
context = ["<s>"]
for i in range(20):
    next_word = model_2.generate_top_k(context, k=4)
    if not next_word: break

    context.append(next_word)
    end = None if next_word != "</s>" else -1
    print(" ".join(context[1: end]))
    clear_output(wait=True)
    time.sleep(0.5)

They study Math and she loves Math


In [ ]:
context = ["<s>"]
for i in range(20):
    next_word = lip_model_2.generate_top_p(context, p=0.75)
    if not next_word: break

    context.append(next_word)
    end = None if next_word != "</s>" else -1
    print(" ".join(context[1: end]))
    clear_output(wait=True)
    time.sleep(0.5)

I love Math


## Corpus 3: Wikipedia Dataset

### Installing the required dependencies

### Loading the Wikipedia dataset

In [43]:
from datasets import load_dataset

In [44]:
dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

### Tokenizing the data into sentences

In [45]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [46]:
from nltk.tokenize import sent_tokenize

def fetch_sentences(dataset):
    for article in dataset:
        text = article["text"]
        if not text.strip(): continue

        sentences = sent_tokenize(text)
        for sentence in sentences:
            cleaned_sentence = sentence.strip()
            yield cleaned_sentence

sentence_stream = fetch_sentences(dataset)

### Building the corpus from 500 randomly picked sentences

In [47]:
import random

wiki_corpus = []
for i, sentence in enumerate(sentence_stream):
    if "\n" not in sentence and random.random() < 0.1:
        wiki_corpus.append(f"<s> {sentence} </s>")
        if len(wiki_corpus) == 500:
            break


### Testing the models

In [56]:
n = 5
model_3 = NGramModel(n, wiki_corpus)
lip_model_3 = LIPNGramModel(n, wiki_corpus, params=[0.8, 0.15, 0.05])

In [57]:
context = ["<s>"]
for i in range(20):
    next_word = model_3.generate_random(context)
    if not next_word: break

    context.append(next_word)
    end = None if next_word != "</s>" else -1
    print(" ".join(context[1: end]))
    clear_output(wait=True)
    time.sleep(0.5)


Prophecies linked Troilus' fate to that of Troy and so he was ambushed in an attempt to capture him.


In [58]:
context = ["<s>"]
for i in range(20):
    next_word = lip_model_3.generate_random(context)
    if not next_word: break

    context.append(next_word)
    end = None if next_word != "</s>" else -1
    print(" ".join(context[1: end]))
    clear_output(wait=True)
    time.sleep(0.5)

Some scholars describe anarchism as having many influences from liberalism, and being both liberal and socialist but more so.
